# Tuples

**Topic:** `3.2` &nbsp;|&nbsp; **Module 3: Core Data Structures**

*Python's record type: an immutable binding of references that unlocks hashable keys, safe unpacking, and the one trap - shallow immutability - that teams discover in production.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M3)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [03_data_structures/README.md](../README.md)

## 1. Why This Topic Matters

Returns of several values, dictionary keys, coordinate grids and mission records are all tuples in real Python systems. The failure students must not carry into industry is trusting 'immutable' data that contains a mutable list - a corrupted audit trail that raises no error at all.

## 2. Learning Objectives

By the end of this topic you will be able to:

1. Construct and unpack tuples, including the one-element comma form.
2. Explain why assignment still aliases immutable objects.
3. Distinguish shallow from deep immutability with a live example.
4. Predict which objects are hashable and use tuples as dict keys.
5. Choose tuple returns over out-parameter mutation for multi-value results.
6. Accumulate into lists and convert once instead of concatenating tuples.

## 3. Prerequisites

This topic assumes you already have:

- Topic 3.1 Lists

## 4. Mental Model

**Mental model: a tuple is a sealed envelope of addresses.** You cannot add, remove or rewrite what is inside the envelope's list of addresses - but if one of those addresses points at a notebook, somebody can still write in the notebook. The envelope is sealed; the notebooks it names are whatever they are.

| Question | Answer | Consequence |
| --- | --- | --- |
| Can I change `t[i]`? | never - the binding is fixed | tuple = record, key, snapshot |
| Can the object at `t[i]` change? | if it is mutable, yes | deep-freeze before trusting it |
| Is `t` hashable? | if every element is | usable as dict key / set member |
| Does `t == u` compare identity? | no - value equality | records compare by content |
| How do I build one incrementally? | build a list, `tuple()` once | avoids quadratic `+` |

## 5. Terminology

Vocabulary used consistently throughout the course.

| Term | Meaning |
| --- | --- |
| Tuple | An ordered, immutable sequence - Python's record type. |
| Immutable binding | The assignment of elements cannot change after creation. |
| Packing | Gathering values into a tuple, usually via commas. |
| Unpacking | Assigning tuple elements to names in one statement. |
| Hashable | An object with a stable hash usable as a dict key or set member. |
| Shallow immutability | The tuple is fixed but its mutable elements are not. |
| Record | A fixed set of fields that logically belong together. |
| Sequence protocol | Indexing, slicing, len, membership and iteration. |
| Lexicographic order | Element-wise comparison that stops at the first difference. |
| Snapshot | A copy of state taken at one instant, isolated from later change. |

## 6. Conceptual Explanation

A tuple is an ordered, **immutable** sequence. Immutability here is a promise about the *binding*: once the tuple is created you cannot append to it, reassign an element, or remove one. That single property is what makes tuples usable as dictionary keys and set members, and it is why functions use them to return several values without inventing a class.

The subtlety that matters in production code is that immutability is shallow. A tuple stores references, exactly like a list, so a tuple that contains a list is immutable in its structure but not in its data: the list inside it can still grow, shrink or be rewritten. "The tuple changed" is almost always this - the binding never changed, one of its referents did.

In [ ]:
pose = (0.0, 0.0, 90.0)
print('pose:', pose)

try:
    pose[0] = 5.0
except TypeError as exc:
    print('assignment rejected:', exc)

record = ([1.0, 2.0], 'calibration')
record[0].append(3.0)          # mutates the LIST inside
print('record after append:', record)

Assignment still aliases: `alias = pose` binds a second name to one tuple. Because the tuple itself cannot change, the aliasing is harmless - but the moment a mutable object sits inside, two names once again share one object, and the list bug from Topic 3.1 returns in tuple's clothing.

Tuples are also the type Python reaches for when data is a *record* - a fixed set of fields that belong together. `(name, x, y)` reads as one thing with three parts; a three-element list reads as three things that happen to be adjacent. Choosing tuple signals "these values are a unit and will not be edited in place", and every reader of your code gets that message for free.

## 7. Formal Theory

The tuple supports the same sequence protocol as a list - indexing, negative indices, slicing, `len`, concatenation, repetition, membership, iteration - with every operation that would mutate removed. What it adds is **hashability**: a tuple whose every element is hashable is itself hashable, and hashability is the price of admission for dictionary keys and set elements.

```text
hashable(tuple) <=> every element is hashable   |   immutable does NOT imply deeply immutable
```

In [ ]:
key = ('rover-01', 3)
registry = {key: 'active'}
print('lookup:', registry[key])

try:
    bad = ([1, 2],)          # list inside - unhashable
    {bad}
except TypeError as exc:
    print('set rejected:', exc)

Two identities keep the model precise. First, tuple construction packs references; it does not copy the elements. Second, equality is by value: two tuples built separately are `==` if their elements are, even though they are different objects (and for small interned ints and short strings they may even be `is`).

| Property | tuple | list |
| --- | --- | --- |
| Ordered | yes | yes |
| Mutable elements | no | yes |
| Hashable (when elements are) | yes | no |
| Suitable as dict key | yes | no |
| Method surface | index, count | 11 mutating + sorted helpers |
| Typical role | record, key, fixed shape | buffer, log, worklist |

Packing and unpacking are one operation seen from two sides: `point = (x, y)` packs; `x, y = point` unpacks. The left side of an unpacking assignment is always a tuple target, which is why `a, b = b, a` swaps without a temporary variable - Python evaluates the right side completely before binding, so no name is overwritten mid-swap.

## 8. Syntax

Construction, unpacking, and the comma that makes a one-element tuple.

In [ ]:
point = (3.0, 4.0)             # two elements - parentheses suffice
single = (42,)                 # ONE element needs the trailing comma
bare = 42,                     # parentheses are optional when packing
empty = ()

print('point :', point)
print('single:', single, ' type:', type(single))
print('bare  :', bare, ' empty:', empty)

x, y = point                   # unpack
print('x =', x, ' y =', y)
x, y = y, x                    # swap via re-packing
print('swapped:', x, y)

The anti-pattern - treating a single-element tuple like a bare value:

```text
not_a_tuple = (42)        # just an int, parentheses were grouping
is_a_tuple = (42,)        # a tuple of one
type(not_a_tuple)         # <class 'int'>
type(is_a_tuple)          # <class 'tuple'>

for item in (42):         # iterates the INT - TypeError
    ...
for item in (42,):        # iterates one item, correctly
    ...
```

> **the missing-comma bug**
>
> `coordinates = (1 2 3)` is a syntax error, but `x = (1)` silently produces an int and `d = {1, 2} (3)` does stranger things still. When a tuple seems to have "lost" an element, check for the forgotten comma before blaming the logic.

## 9. Basic Examples

**Example 1 - a coordinate record and its unpacking.**

In [ ]:
origin = (0.0, 0.0)
target = (3.0, 4.0)

dx, dy = target[0] - origin[0], target[1] - origin[1]
print('delta:', (dx, dy))

for label, value in (('dx', dx), ('dy', dy)):
    print(f'{label} = {value}')

**Example 2 - returning several values without a class.**

In [ ]:
def bounds(values):
    """Return (minimum, maximum, span) of a non-empty sequence."""
    lo, hi = min(values), max(values)
    return lo, hi, hi - lo        # implicitly packed into a tuple

lo, hi, span = bounds([4, 1, 9, 2])
print('lo:', lo, ' hi:', hi, ' span:', span)

**Example 3 - tuples as dictionary keys, the capability lists have.**

In [ ]:
route = {}
route[('alpha', 'beta')] = 12.5
route[('alpha', 'gamma')] = 8.0

for (start, end), metres in sorted(route.items()):
    print(f'{start} -> {end}: {metres} m')

## 10. Intermediate Examples

**Named fields without a class.** Unpacking by position is clear while the shape is stable; the moment a record grows a field, positional unpacking breaks everywhere at once - the signal to reach for a namedtuple or a class.

In [ ]:
def locate(readings):
    best = max(readings, key=lambda r: r[1])
    return best[0], best[1], readings.index(best)

tag, value, idx = locate([('a', 0.4), ('b', 0.9), ('c', 0.7)])
print(f'peak {tag} = {value} at index {idx}')

**Tuple comprehensions do not exist** - `(... for x in ...)` is a generator expression, not a tuple. Materialise deliberately with `tuple(...)`, and notice the memory difference: a tuple built eagerly costs exactly as much as the list would.

In [ ]:
squares_gen = (i * i for i in range(5))      # a GENERATOR, not a tuple
print('generator type:', type(squares_gen))

squares_tup = tuple(i * i for i in range(5))  # eager materialisation
print('tuple:', squares_tup, ' len:', len(squares_tup))
print('walked once:', list(squares_gen))      # still full - unvisited

**Fixed-size fields favour tuples in dataclass-like returns.** `min`, `max` and `divmod` all return tuples for exactly this reason: the shape is documented, the values cannot be edited in place, and callers can unpack or index.

In [ ]:
quotient, remainder = divmod(17, 5)
print('divmod(17, 5):', quotient, remainder)

first, *middle, last = (1, 2, 3, 4)
print('star unpack:', first, middle, last)

## 11. Advanced Examples

**A tuple that is not immutable.** The binding is fixed; the list inside it is not. This is the failure that surprises teams most, because the type checker and the `try` block both say "immutable" while the data changes under a log entry.

In [ ]:
entry = ([0.4, 0.9], 'raw')
before = entry
entry[0].append(0.7)              # legal - mutates the inner list
print('same entry object?', entry is before)
print('content changed  :', entry[0])

frozen = (tuple([0.4, 0.9]), 'raw')
try:
    frozen[0].append(0.7)
except AttributeError as exc:
    print('deep freeze holds:', exc)

**Hashability decides keys.** A tuple is hashable only if every element is. One list, one dict, or one set anywhere inside makes the whole tuple unhashable - and the TypeError names the inner type, which is the fastest debugging handle you have for this class of bug.

In [ ]:
good = ('rover-01', (3.0, 4.0), 90.0)
print('hashable:', hash(good) is not None)

bad = ('rover-01', [3.0, 4.0])
try:
    hash(bad)
except TypeError as exc:
    print('unhashable:', exc)

frozen_bad = ('rover-01', (3.0, 4.0))   # inner tuple: hashable again
print('fixed by inner tuple:', hash(frozen_bad) is not None)

> **freeze before keying**
>
> To use structured data as a key, convert containers to tuples recursively - or store a `json.dumps(obj, sort_keys=True)` string when the structure is deep. Both trades are explicit; both are testable.

## 12. Code Walkthrough

**A calibration record that cannot drift.** Sensor calibration is a fixed record: offset, scale, timestamp. Returning it as a tuple makes accidental in-place edits impossible, and validating on construction catches bad data at the boundary rather than deep in a pipeline.

In [ ]:
def make_calibration(offset, scale, timestamp):
    """Return a validated (offset, scale, timestamp) record."""
    if not isinstance(timestamp, (int, float)):
        raise TypeError('timestamp must be numeric')
    if scale == 0:
        raise ValueError('scale must be non-zero')
    return (float(offset), float(scale), float(timestamp))

cal = make_calibration(0.02, 1.5, 1712000000.0)
print('calibration:', cal)
offset, scale, ts = cal
print('corrected  :', round((1.0 * scale) + offset, 4))

- The tuple's fixed shape is the function's documented contract - three fields, always in this order.
- Validation runs before packing, so a bad record never escapes the constructor.
- `float(...)` normalises types once, at the boundary, so downstream math never sees a string.
- Callers may unpack (`offset, scale, ts = cal`) or index (`cal[1]`) - both are stable because the shape never changes.

## 13. Line-by-Line Explanation

1. `def make_calibration(offset, scale, timestamp):` - three scalar parameters; the tuple is built only at the return.
2. `if not isinstance(timestamp, (int, float)):` - rejects strings before any arithmetic can fail later with a confusing TypeError.
3. `if scale == 0:` - a zero scale would silently erase every corrected reading, so it must fail loudly here.
4. `return (float(offset), float(scale), float(timestamp))` - one packing expression; the parentheses document the record's shape.
5. `offset, scale, ts = cal` - unpacking is by position; if the return shape ever changed, this line is where it would break, loudly and locally.
6. `round((1.0 * scale) + offset, 4)` - the corrected value uses both fields, proving the record carries everything a consumer needs.

## 14. Common Mistakes

**Mistake 1 - forgetting the comma on a one-element tuple.**

```text
a = (42)          # int 42
b = (42,)         # tuple of one
c = 42,           # also a tuple of one - parentheses are optional
print(type(a), type(b), type(c))
```

**Mistake 2 - expecting a tuple's contents to be immutable.**

```text
config = ([1, 2], 'mode')
config[0].append(3)     # succeeds - the LIST inside changed
print(config)           # ([1, 2, 3], 'mode')

safe = (tuple([1, 2]), 'mode')
# safe[0].append(3)     # AttributeError: 'tuple' object has no attribute
```

**Mistake 3 - using a list where a key is needed.**

```text
route = {}
route[['a', 'b']] = 12     # TypeError: unhashable type: 'list'
route[('a', 'b')] = 12     # tuples are the key type
```

**Mistake 4 - building a 'tuple comprehension' that is a generator.**

```text
squares = (i * i for i in range(4))   # a generator object!
print(len(squares))                   # TypeError
squares = tuple(i * i for i in range(4))   # the tuple you meant
```

**Mistake 5 - unpacking the wrong number of values.**

```text
point = (1, 2, 3)
x, y = point            # ValueError: too many values to unpack
x, y, z = point         # correct: three targets for three values
x, *rest = point        # correct: absorb the tail
```

## 15. Debugging Techniques

When a tuple is rejected as a key, print `type()` of each element - the innermost unhashable element is always the culprit, and the error message names it if you let it raise fully.

In [ ]:
record = ('rover-01', [3.0, 4.0], 90.0)
for i, item in enumerate(record):
    try:
        hash(item)
        print(f'element {i}: {type(item).__name__} is hashable')
    except TypeError as exc:
        print(f'element {i}: {type(item).__name__} FAILS - {exc}')

When "immutable" data seems to change, capture `id()` before and after, and check whether the tuple object itself moved (it should not) or only its contents (an inner mutable did).

In [ ]:
entry = ([1.0], 'raw')
before_id, before_content = id(entry), list(entry[0])
entry[0].append(2.0)

print('tuple object moved?', id(entry) != before_id)     # False
print('contents changed ?', entry[0] != before_content)  # True
print('diagnosis: inner list mutated, tuple binding intact')

> **Unpacking ValueError messages are exact**
>
> `too many values to unpack` and `not enough values to unpack` tell you the shape on the left and right disagree - print `len()` of both sides instead of re-reading the code by eye.

## 16. Best Practices

- Use a tuple for records with a fixed field set; use a list for data that will grow or be reordered.
- Always write the trailing comma: `(x,)` - and review any `(x)` for the same reason.
- Return `a, b` (bare comma) from functions - it reads as multiple values and packs itself.
- Never rely on a tuple for deep immutability; freeze inner containers if you need real read-only data.
- Prefer unpacking to indexing when the shape is fixed and known - `x, y, z = record` documents the arity.
- Reach for `collections.namedtuple` or a dataclass before a tuple exceeds about three positional fields.
- Convert lists to tuples before using them as dict keys or set members, and test that conversion with `hash()`.
- Do not build tuples with comprehensions - use `tuple(gen)` and know you are materialising eagerly.

## 17. Performance Considerations

Tuples are marginally smaller and faster than lists - they are allocated once with an exact size, while lists over-allocate to make appends cheap. The gap is real but small (typically 10-20% on iteration); the *architectural* win is larger: a hashable tuple can be a key, which turns a linear scan into a hash lookup.

| Operation | tuple | list |
| --- | --- | --- |
| iteration | slightly faster (no over-allocation bookkeeping) | baseline |
| membership `x in t` | O(n) - same as list | O(n) |
| as dict key | O(1) lookup after hashing | TypeError - not allowed |
| append during build | impossible - rebuild with + (O(n) each) | amortised O(1) |
| memory per element | lower (exact allocation) | higher (spare slots) |

In [ ]:
import sys

data_list = list(range(1000))
data_tuple = tuple(range(1000))

print('list bytes  :', sys.getsizeof(data_list))
print('tuple bytes :', sys.getsizeof(data_tuple))

index = {value: i for i, value in enumerate(data_tuple)}
print('tuple as key: index[999] =', index[999])

One pattern genuinely is quadratic with tuples: accumulating with `acc = acc + (item,)` inside a loop recopies the whole tuple every pass. If you are building a sequence incrementally, build a list and `tuple()` it once at the end - that is the idiom in standard-library code for exactly this reason.

In [ ]:
import time

n = 8000
start = time.perf_counter()
acc_t = ()
for i in range(n):
    acc_t = acc_t + (i,)          # recopies every pass
t_concat = time.perf_counter() - start

start = time.perf_counter()
acc_l = []
for i in range(n):
    acc_l.append(i)
acc_t2 = tuple(acc_l)            # one materialisation at the end
t_list = time.perf_counter() - start

print(f'tuple += {t_concat:.4f}s vs list+tuple() {t_list:.4f}s')
print('equal result?', acc_t == acc_t2)

## 18. Pythonic Approaches

Idiomatic tuple use is about *signalling*: bare-comma returns, unpacking instead of indexing, and swapping without a temporary - all of which tell the reader what the data's shape means.

In [ ]:
def centroid(points):
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    return sum(xs) / len(xs), sum(ys) / len(ys)   # bare-comma return

cx, cy = centroid([(0.0, 0.0), (3.0, 4.0)])
print(f'centroid: ({cx:.2f}, {cy:.2f})')

left, right = 'stop', 'go'       # swap - no temp variable
left, right = right, left
print('swapped:', left, right)

- Return `a, b` rather than `return (a, b)` - both pack, the first reads as "two values".
- Unpack on assignment (`x, y = record`) instead of indexing when the arity is fixed.
- Use `_` for unpacked values you intend to ignore: `_, status = probe()`.
- `a, b = b, a` beats a temporary variable for swaps - Python evaluates the whole right side first.
- Iterate tuple fields with zip: `for name, value in zip(names, values)` pairs records without indices.
- Wrap a generator with `tuple(...)` explicitly when you need random access or a hashable snapshot.

## 19. Robotics Connection

A pose - position and heading - is the canonical robotics record. It never grows mid-run, every subsystem needs it, and keys like `(cell_x, cell_y)` turn occupancy grids into dictionaries. Tuples are the natural currency of that data.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / 'shared').is_dir()), None)
if root is not None:
    sys.path.insert(0, str(root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402

robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
robot.move(2.0, speed_mps=0.5)

pose = tuple(robot.status()['position'])          # (x, y) snapshot
cell = (round(pose[0]), round(pose[1]))           # hashable grid key
visited = {cell: 'explored'}

print('pose :', pose)
print('cell :', cell, '->', visited[cell])
print('battery:', round(robot.status()['battery_pct'], 1), '%')

Two habits make this safe in practice. First, snapshot mutable state into a tuple at the boundary (`tuple(status['position'])`) so later simulator ticks cannot rewrite history held by the planner. Second, derive keys as tuples of primitives - `(cell_x, cell_y)` - so the visited set stays hashable no matter how the pose type evolves.

## 20. Engineering Example

Version comparisons are tuple comparisons: Python compares sequences lexicographically, element by element, which makes `(major, minor, patch)` ordering correct with no parsing code at all - provided the numeric fields are really numeric.

In [ ]:
def parse_version(text):
    """'3.12.1' -> (3, 12, 1); malformed parts raise ValueError."""
    parts = text.strip().split('.')
    if len(parts) != 3:
        raise ValueError(f'expected major.minor.patch, got {text!r}')
    return tuple(int(p) for p in parts)

current = parse_version('3.12.1')
minimum = (3, 10, 0)
print('current:', current, ' meets minimum?', current >= minimum)
print('(3, 9, 99) < current?', (3, 9, 99) < current)

The tuple comparison does the work: it scans left to right and stops at the first difference, so `(3, 12, 1) >= (3, 10, 0)` is decided at the second element. The one trap is comparing across types - a string field compared against an int raises TypeError - which is exactly why `parse_version` converts every part before the tuple is built.

## 21. Guided Practice

1. Create a one-element tuple two ways - `(42,)` and `42,` - and print both types to confirm they match.
2. Unpack a three-field record with `a, b, c = record`, then unpack again using `a, *rest` and compare the results.
3. Use a tuple of two ints as a dictionary key and prove a list of the same values raises TypeError.
4. Build a tuple containing a list, mutate the list, and explain using the words 'binding' and 'referent' why nothing about the tuple itself changed.
5. Swap two names without a temporary, then time `acc = acc + (i,)` against list-append plus `tuple()` for n = 5000.

In [ ]:
single_a = (42,)
single_b = 42,                 # same one-element tuple, no parens
print('types:', type(single_a).__name__, type(single_b).__name__)
print('equal:', single_a == single_b)

record = ('alpha', 0.4, 1712000000)
first, *rest = record
print('first:', first, ' rest:', rest)

registry = {('cell', 3): 'occupied'}
try:
    {['cell', 3]}
except TypeError as exc:
    print('list rejected:', exc)

container = ([1.0], 'raw')
container[0].append(2.0)
print('binding intact, contents now:', container)

## 22. Knowledge Checks

Answer these before moving on. Full answers are in [solution.ipynb](solution.ipynb).

- **Q1.** What is the type of `(42)` versus `(42,)`?
- **Q1.** Why does `hash(('a', [1, 2]))` raise TypeError?
- **Q1.** Why are coordinate tuples preferred over lists as occupancy-grid keys?
- **Q1.** Which version comparison is correct, and why?

## 23. Summary

A tuple is an ordered sequence with a sealed binding: once created, its elements cannot be added, removed or replaced. That makes it the record type of the language - `(name, x, y)` states its own shape - and, when every element is hashable, the key type too: dictionary keys and set members are tuples' natural habitat.

Immutability is shallow, which is the one fact that must survive the lesson. A tuple holding a list is fixed in structure and open in content, so 'the tuple changed' almost always means a referent did. Freeze inner containers with `tuple(...)` when genuine read-only data is required, and test it with `hash()` rather than trusting the type name.

Operationally, tuples are cheap to iterate and impossible to extend - accumulate into a list and convert once, never `acc = acc + (x,)`. Unpack instead of index while the arity is fixed, switch to a namedtuple or dataclass the moment a record outgrows three fields, and snapshot mutable state into tuples at module boundaries so history cannot be rewritten under a planner's feet.

## 24. Key Takeaways

- Tuples are ordered and immutable in their binding; assignment still aliases.
- One element needs a comma: `(42,)` - `(42)` is just 42.
- Immutability is shallow: a tuple containing a list is not deeply immutable.
- Hashable tuples (all elements hashable) become dict keys and set members; lists cannot.
- Unpack by position for fixed records; index when the arity is variable.
- There is no tuple comprehension - `tuple(gen)` materialises eagerly.
- Accumulate with a list, convert once; `acc + (x,)` in a loop is quadratic.
- `a, b = b, a` swaps safely because the right side is fully evaluated before binding.

## 25. Further Exploration

- Read about `collections.namedtuple` and compare it with a plain tuple at three fields and at six.
- Investigate why `sys.getsizeof((1, 2, 3))` beats the equivalent list - over-allocation is the answer.
- Time `acc = acc + (i,)` against list-append plus `tuple()` for n = 10^4, 10^5 and confirm the quadratic curve.
- Explore tuple equality across types: what does `(1, 2) == (1.0, 2.0)` return, and why does that matter for version keys?
- Look up PEP 435 (enum) to see how Python formalised the 'immutable record' idea.